<a href="https://colab.research.google.com/github/TANISHQ-28/FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TANISHQ-28/FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### ANSWER :    
Plain Words Rule: I wanted to catch pages that show up on the first page of Google (so average position is 10 or better), but are totally failing to get clicks. Basically, if a page has decent impressions (at least 50) and a good rank, but its CTR is under 1%, something's wrong with the title or snippet.

 Reason Codes: UNDERPERFORMING_CTR: High rank, low clicks.HEALTHY_SIGNAL: Everything else that looks normal.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Code :
import duckdb
import os
import getpass
import pandas as pd

# 1. Safely prompt for your Hugging Face token if it hasn't been defined yet
if 'HF_TOKEN' not in globals():
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

# 2. Connect DuckDB and authenticate using the token
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("--- Successfully Connected to FlyRank Warehouse ---")

## Actual CODE PART for the assignment.

# 3. Quick check to see how many rows actually trigger this rule
rule_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_avg_position <= 10 AND (gsc_clicks::FLOAT / NULLIF(gsc_impressions, 0)) < 0.01 AND gsc_impressions >= 50 THEN 1 ELSE 0 END) AS flagged_count
    FROM {TABLES['fact_daily_sample']}
""").fetchone()

print(f"Scanned {rule_check[0]:,} rows. Found {rule_check[1]:,} pages hitting the underperforming CTR flag.")

Paste your Hugging Face READ token (hf_...): ··········
--- Successfully Connected to FlyRank Warehouse ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scanned 11,694,072 rows. Found 460,623 pages hitting the underperforming CTR flag.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### ANSWER :    
Running the query to score everything based on impression volume and ranking position, sorting them out, and saving the final output file straight to work/outputs/baseline_action_score.csv as required.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Code :

os.makedirs("work/outputs", exist_ok=True)

df_queue = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) ELSE 0 END AS gsc_ctr,
        CASE
            WHEN gsc_avg_position <= 10 AND (gsc_clicks::FLOAT / NULLIF(gsc_impressions, 0)) < 0.01 AND gsc_impressions >= 50
            THEN 'UNDERPERFORMING_CTR'
            ELSE 'HEALTHY_SIGNAL'
        END AS reason_code,
        (gsc_impressions * (11 - LEAST(gsc_avg_position, 10))) AS priority_score
    FROM {TABLES['fact_daily_sample']}
    ORDER BY priority_score DESC
    LIMIT 100
""").df()

output_path = "work/outputs/baseline_action_score.csv"
df_queue.to_csv(output_path, index=False)
print(f"Saved the ranked queue to {output_path} with {len(df_queue)} rows.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved the ranked queue to work/outputs/baseline_action_score.csv with 100 rows.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### ANSWER :    
Top-20 Review Analysis:

Action: Rewrite meta tags and titles for these pages since they already rank high but lack user engagement.

Reason Code: UNDERPERFORMING_CTR

Confidence Note: Pretty confident since we're just using straightforward GSC position stats without peeking into the future.  

What Would Make It Wrong: If it's a query where people just look at the Google answer box and don't need to click through, a low CTR doesn't necessarily mean the page is broken.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# CODE :
# Let's peek at the top entries
df_top20 = pd.read_csv("work/outputs/baseline_action_score.csv").head(20)
print(df_top20[['content_hash_id', 'gsc_avg_position', 'gsc_ctr', 'reason_code', 'priority_score']].head(10))


            content_hash_id  gsc_avg_position   gsc_ctr          reason_code  \
0  content_963de14b1f58978f          6.326149  0.003620  UNDERPERFORMING_CTR   
1  content_eadb33b5df496f4a          2.627995  0.003504  UNDERPERFORMING_CTR   
2  content_eadb33b5df496f4a          2.590467  0.003858  UNDERPERFORMING_CTR   
3  content_545bb6cc7081ded3          2.586277  0.002329  UNDERPERFORMING_CTR   
4  content_545bb6cc7081ded3          2.605735  0.002519  UNDERPERFORMING_CTR   
5  content_545bb6cc7081ded3          2.605150  0.003142  UNDERPERFORMING_CTR   
6  content_eadb33b5df496f4a          2.513816  0.004628  UNDERPERFORMING_CTR   
7  content_eadb33b5df496f4a          2.097107  0.006449  UNDERPERFORMING_CTR   
8  content_545bb6cc7081ded3          1.931900  0.002668  UNDERPERFORMING_CTR   
9  content_f88878f155e4838d          5.828089  0.007169  UNDERPERFORMING_CTR   

   priority_score  
0       1148954.0  
1        413351.0  
2        411983.0  
3        411877.0  
4        356538.0  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### ANSWER :      
Weak Picks Critique: A few low-traffic pages snuck up higher in priority just because they had an awesome position number, even though their raw impression numbers were tiny. Might need to weight impression volume a bit heavier next time.

Leakage & Privacy Check: Checked the CSV columns—no target data, no future windows, and no private URLs leaked in. Everything looks clean.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# CODE :
csv_check = pd.read_csv("work/outputs/baseline_action_score.csv")
bad_cols = [c for c in csv_check.columns if 'future' in c.lower() or 'target' in c.lower()]
print(f"Leaky columns found: {bad_cols} (should be empty)")
print("Leakage check passed.")


Leaky columns found: [] (should be empty)
Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.